In [ ]:
using Revise

ENV["PYCALL_JL_RUNTIME_PYTHON"] = Sys.which("python")
using FileIO
using JLD2
using PyCall
pyimport("sys")."path" |> x -> pushfirst!(x, "../Python-RVO2/build/lib.linux-x86_64-3.6")

push!(LOAD_PATH,"../src")
using RiskSensitiveSAC

In [ ]:
include("$(@__DIR__)/../scripts/default_params/params_synthetic_gaussian.jl");

include("$(@__DIR__)/../scripts/parameter_setup.jl");

In [ ]:
scene_loader, controller, w_init, measurement_schedule, target_trajectory, target_speed = 
controller_setup(scene_param,
                 predictor_param,
                 prediction_device=prediction_device,
                 cost_param=cost_param,
                 cnt_param=cnt_param,
                 dtc=dtc,
                 dto=dto,
                 ado_pos_init_dict=ado_pos_init_dict,
                 ado_vel_dict=ado_vel_dict,
                 ego_pos_init_vec=ego_pos_init_vec,
                 ego_pos_goal_vec=ego_pos_goal_vec,
                 target_speed=target_speed,
                 sim_horizon=sim_horizon,
                 verbose=false);

In [ ]:
result, ~, ~ = evaluate(scene_loader, controller, w_init, ego_pos_goal_vec,
                  target_speed, measurement_schedule, target_trajectory,
                  pos_error_replan, nominal_control=false);

In [ ]:
display_log(result.log)

In [ ]:
result.total_cnt_cost

In [ ]:
result.total_pos_cost

In [ ]:
result.total_col_cost

In [ ]:
result.total_cnt_cost + result.total_pos_cost + result.total_col_cost

In [ ]:
minimum([minimum(vcat([norm(get_position(w.e_state) - ap) for ap in values(w.ap_dict)], Inf))
                          for w in result.w_history])

In [ ]:
make_gif(result, dtplot=0.02, fps=50, xlim=(-6., 6.), ylim=(-6., 6.), figsize=(600, 400), 
         legendfontsize=7, legend=:topright, markersize=5., filename="1_synthetic_gaussian.gif",
         show_nominal_trajectory=false)

In [ ]:
save("1_synthetic_gaussian.jld2", "result", result)